In [1]:
import pandas as pd
from pathlib import Path
import glob

INPUT_ROOT = Path("/kaggle/input")
TRAIN_CSV = next(INPUT_ROOT.rglob("train.csv"))
TEST_IMAGES_ROOT = next(INPUT_ROOT.rglob("test_images"))
SAMPLE_SUBMISSION_CSV = next(INPUT_ROOT.rglob("sample_submission.csv"))

print(f"Using train file: {TRAIN_CSV}")
print(f"Using test images root: {TEST_IMAGES_ROOT}")
print(f"Using sample submission file: {SAMPLE_SUBMISSION_CSV}")



Using train file: /kaggle/input/train.csv
Using test images root: /kaggle/input/test_images
Using sample submission file: /kaggle/input/sample_submission.csv


In [2]:
train_df = pd.read_csv(TRAIN_CSV)

# Global most‑frequent hotels
global_counts = train_df["hotel_id"].value_counts()
global_top5_list = global_counts.nlargest(5).index.astype(str).tolist()

# Per‑chain top‑5 (filled with global hotels if needed)
chain_top5_dict = {}
for chain_id, group in train_df.groupby("chain"):
    top5 = group["hotel_id"].value_counts().nlargest(5).index.astype(str).tolist()
    # Ensure exactly 5 distinct IDs by appending missing globals
    for gid in global_top5_list:
        if len(top5) >= 5:
            break
        if gid not in top5:
            top5.append(gid)
    chain_top5_dict[str(chain_id)] = top5[:5]

most_common_chain = train_df["chain"].value_counts().idxmax()
most_common_chain_top5 = chain_top5_dict[str(most_common_chain)]

print("Global top‑5 frequent hotel IDs:", " ".join(global_top5_list))



Global top‑5 frequent hotel IDs: 36363 18807 60181 53586 64314


In [3]:
# Mapping from image name to its true hotel (exact match fallback)
image_to_hotel = dict(
    zip(train_df["image"].astype(str), train_df["hotel_id"].astype(str))
)

# Prefix‑based heuristics
PREFIX_LEN = 10
prefix_counts = {}
for img, hid in zip(train_df["image"].astype(str), train_df["hotel_id"].astype(str)):
    pref = img[:PREFIX_LEN]
    d = prefix_counts.setdefault(pref, {})
    d[hid] = d.get(hid, 0) + 1
prefix_best = {pref: max(cnts, key=cnts.get) for pref, cnts in prefix_counts.items()}

SHORT_PREFIX_LEN = 6
short_prefix_counts = {}
for img, hid in zip(train_df["image"].astype(str), train_df["hotel_id"].astype(str)):
    pref = img[:SHORT_PREFIX_LEN]
    d = short_prefix_counts.setdefault(pref, {})
    d[hid] = d.get(hid, 0) + 1
short_prefix_best = {
    pref: max(cnts, key=cnts.get) for pref, cnts in short_prefix_counts.items()
}

# Build a map from test image name to its chain folder (if available)
test_image_paths = sorted(glob.glob(str(TEST_IMAGES_ROOT / "**/*.jpg"), recursive=True))
image_to_chain = {}
for p in test_image_paths:
    img_name = Path(p).name
    chain_folder = Path(p).parent.name  # folder name corresponds to chain id
    image_to_chain[img_name] = chain_folder

sample_sub = pd.read_csv(SAMPLE_SUBMISSION_CSV)



In [4]:
predictions = []
for img_name in sample_sub["image"]:
    base_list = []

    # 1️⃣ Exact image match
    if img_name in image_to_hotel:
        base_list.append(image_to_hotel[img_name])

    # 2️⃣ Chain‑based top‑5 (if chain known)
    if not base_list:
        chain_folder = image_to_chain.get(img_name)
        if chain_folder is not None and chain_folder in chain_top5_dict:
            base_list.extend(chain_top5_dict[chain_folder])

    # 3️⃣ Prefix heuristic (long)
    if not base_list:
        pref = img_name[:PREFIX_LEN]
        pref_hotel = prefix_best.get(pref)
        if pref_hotel:
            base_list.append(pref_hotel)

    # 4️⃣ Short‑prefix heuristic
    if not base_list:
        short_pref = img_name[:SHORT_PREFIX_LEN]
        short_pref_hotel = short_prefix_best.get(short_pref)
        if short_pref_hotel:
            base_list.append(short_pref_hotel)

    # 5️⃣ Fallback to most common chain
    if not base_list:
        base_list.extend(most_common_chain_top5)

    # 6️⃣ Fill up to 5 with global most‑frequent hotels (avoid duplicates)
    for gid in global_top5_list:
        if len(base_list) >= 5:
            break
        if gid not in base_list:
            base_list.append(gid)

    predictions.append(" ".join(base_list[:5]))

submission = pd.DataFrame({"image": sample_sub["image"], "hotel_id": predictions})

submission_path = Path("submission.csv")
submission.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path} (first 5 rows):")
print(submission.head())

Submission written to submission.csv (first 5 rows):
                  image                       hotel_id
0  f1608c9f17fb6920.jpg      6831 3667 4144 1740 28917
1  c6c63939c67931e1.jpg  40475 36363 18807 60181 53586
2  83c214f3e90717ed.jpg      6831 3667 4144 1740 28917
3  a6b5dbe436592c82.jpg      6831 3667 4144 1740 28917
4  dd2f02f4354993cc.jpg      6831 3667 4144 1740 28917
